In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/ColabNotebooks/'
df = pd.read_csv(f'{base_path}cohort_csv/data_after_outlier.csv')

Mounted at /content/drive


In [ ]:
# =============================================================================
# 2. 변수 그룹 정의
# =============================================================================
print("\n" + "=" * 80)
print("[2] 변수 그룹 정의")
print("=" * 80)

# Group A: 저결측 (<30%) - Median Imputation
group_A_numeric = [
    'WBC', 'HB', 'PLATELET', 'CREATININE', 'BUN',
    'SODIUM', 'POTASSIUM', 'CHLORIDE', 'BICARBONATE', 'GLUCOSE_LAB',
    'ANION_GAP_APPROX', 'BUN_CR_RATIO',
    'AGE_AT_ADMISSION'
]

group_A_categorical = ['GENDER'] # M=1, F=0

# Group B: 중결측 (30-70%) - MICE
group_B = [
    'LACTATE', 'CALCIUM', 'BILIRUBIN',
    'INR', 'PT', 'PTT',
    'URINE_SPEC_GRAVITY',
    'ALBUMIN'  # ✅ 추가!
]

# Group C: 고결측 Vital Signs (>70%) - Median + Indicator
group_C_vital = [
    'HEART_RATE', 'SBP', 'DBP', 'MAP',
    'RESP_RATE', 'TEMP', 'SPO2'
]

# Group D: 고결측 특수검사 (>70%) - Indicator만 (값 사용 안 함)
group_D_special = [
    'CRP', 'PAO2', 'FIO2', 'GCS', 'SHOCK_INDEX'
]

# Group E: 제외 변수 (Indicator만)
group_E_exclude = [
    'GLUCOSE_CHART',
    'TOTAL_URINE_OUTPUT',
    'URINE_PROTEIN'
]

print(f"\n변수 그룹:")
print(f"  Group A (저결측 <30%): {len(group_A_numeric)}개 → Median")
print(f"  Group B (중결측 30-70%): {len(group_B)}개 → MICE")
print(f"  Group C (Vital Signs >70%): {len(group_C_vital)}개 → Median + Indicator")
print(f"  Group D (특수검사 >70%): {len(group_D_special)}개 → Indicator만 ⚠️")
print(f"  Group E (제외): {len(group_E_exclude)}개 → Indicator만")



[2] 변수 그룹 정의

변수 그룹:
  Group A (저결측 <30%): 13개 → Median
  Group B (중결측 30-70%): 8개 → MICE
  Group C (Vital Signs >70%): 7개 → Median + Indicator
  Group D (특수검사 >70%): 5개 → Indicator만 ⚠️
  Group E (제외): 3개 → Indicator만


In [ ]:
# =============================================================================
# 3. Target 변수 분리
# =============================================================================
print("\n" + "=" * 80)
print("[3] Target 변수 분리")
print("=" * 80)

# 패혈증
y_sepsis = df['SEPSIS'].copy()
print(f"패혈증 발생률: {y_sepsis.mean()*100:.2f}% ({y_sepsis.sum():,}명 / {len(y_sepsis):,}명)")

# 사망 (있다면)
if 'HOSPITAL_EXPIRE_FLAG' in df.columns:
    y_mortality = df['HOSPITAL_EXPIRE_FLAG'].copy()
    print(f"사망률: {y_mortality.mean()*100:.2f}% ({y_mortality.sum():,}명 / {len(y_mortality):,}명)")
else:
    y_mortality = None
    print("사망 데이터 없음")

# Feature 데이터프레임
df_features = df.copy()



[3] Target 변수 분리
패혈증 발생률: 6.17% (2,662명 / 43,171명)
사망률: 4.07% (1,758명 / 43,171명)


In [ ]:
# =============================================================================
# 4. Train/Test Split
# =============================================================================
print("\n" + "=" * 80)
print("[4] Train/Test Split (8:2)")
print("=" * 80)

# 패혈증 기준으로 split
train_idx, test_idx = train_test_split(
    range(len(df)),
    test_size=0.2,
    stratify=y_sepsis,
    random_state=42
)

df_train = df_features.iloc[train_idx].copy()
df_test = df_features.iloc[test_idx].copy()

y_train_sepsis = y_sepsis.iloc[train_idx]
y_test_sepsis = y_sepsis.iloc[test_idx]

if y_mortality is not None:
    y_train_mortality = y_mortality.iloc[train_idx]
    y_test_mortality = y_mortality.iloc[test_idx]

print(f"Train: {len(df_train):,}명 (패혈증: {y_train_sepsis.sum():,}명, {y_train_sepsis.mean()*100:.2f}%)")
print(f"Test: {len(df_test):,}명 (패혈증: {y_test_sepsis.sum():,}명, {y_test_sepsis.mean()*100:.2f}%)")



[4] Train/Test Split (8:2)
Train: 34,536명 (패혈증: 2,130명, 6.17%)
Test: 8,635명 (패혈증: 532명, 6.16%)


In [ ]:
# =============================================================================
# 5. Indicator 변수 생성
# =============================================================================
print("\n" + "=" * 80)
print("[5] Indicator 변수 생성")
print("=" * 80)

def create_indicators(df):
    """
    고결측 변수에 대한 측정 여부 이진 변수 생성
    """
    df = df.copy()

    # Group C: Vital Signs Indicator
    for col in group_C_vital:
        if col in df.columns:
            df[f'{col}_measured'] = (~df[col].isnull()).astype(int)

    # Group D: 특수검사 Indicator
    for col in group_D_special:
        if col in df.columns:
            df[f'{col}_measured'] = (~df[col].isnull()).astype(int)

    # Group E: 제외 변수 Indicator
    for col in group_E_exclude:
        if col in df.columns:
            df[f'{col}_measured'] = (~df[col].isnull()).astype(int)

    return df

df_train = create_indicators(df_train)
df_test = create_indicators(df_test)

print(f"✅ Indicator 생성 완료:")
print(f"   - Vital Signs: {len(group_C_vital)}개")
print(f"   - 특수검사: {len(group_D_special)}개")
print(f"   - 제외 변수: {len(group_E_exclude)}개")
print(f"   - 총 {len(group_C_vital) + len(group_D_special) + len(group_E_exclude)}개 indicator 생성")



[5] Indicator 변수 생성
✅ Indicator 생성 완료:
   - Vital Signs: 7개
   - 특수검사: 5개
   - 제외 변수: 3개
   - 총 15개 indicator 생성


In [ ]:
# =============================================================================
# 6. 결측치 대체 (FIXED VERSION)
# =============================================================================
print("\n" + "=" * 80)
print("[6] 결측치 대체")
print("=" * 80)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 6-1. Group A (Numeric): Median Imputation
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n[6-1] Group A (수치형) - Median Imputation")

group_A_num_exists = [c for c in group_A_numeric if c in df_train.columns]
print(f"사용 가능한 변수: {len(group_A_num_exists)}개 / {len(group_A_numeric)}개")

if group_A_num_exists:
    imputer_A_num = SimpleImputer(strategy='median')

    imputer_A_num.fit(df_train[group_A_num_exists])

    df_train[group_A_num_exists] = imputer_A_num.transform(df_train[group_A_num_exists])
    df_test[group_A_num_exists] = imputer_A_num.transform(df_test[group_A_num_exists])

    print(f"✅ 수치형 {len(group_A_num_exists)}개 처리 완료")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 6-1-b. Group A (Categorical): GENDER 처리
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n[6-1-b] Group A (범주형) - GENDER Encoding")

if 'GENDER' in df_train.columns:
    gender_map = {'M': 1, 'F': 0}

    df_train['GENDER'] = df_train['GENDER'].map(gender_map)
    df_test['GENDER'] = df_test['GENDER'].map(gender_map)

    # 혹시 모를 결측 → 최빈값
    mode_gender = df_train['GENDER'].mode()[0]
    df_train['GENDER'] = df_train['GENDER'].fillna(mode_gender)
    df_test['GENDER'] = df_test['GENDER'].fillna(mode_gender)

    print("✅ GENDER: M=1, F=0 + mode imputation 완료")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 6-2. Group B: MICE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n[6-2] Group B (중결측 30-70%) - MICE")

group_B_exists = [c for c in group_B if c in df_train.columns]
print(f"사용 가능한 변수: {len(group_B_exists)}개 / {len(group_B)}개")

if group_B_exists:
    imputer_B = IterativeImputer(
        max_iter=10,
        random_state=42
    )

    print("⏳ MICE 학습 중...")
    imputer_B.fit(df_train[group_B_exists])

    df_train[group_B_exists] = imputer_B.transform(df_train[group_B_exists])
    df_test[group_B_exists] = imputer_B.transform(df_test[group_B_exists])

    print(f"✅ {len(group_B_exists)}개 변수 처리 완료")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 6-3. Group C: Vital Signs (Median + Indicator)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n[6-3] Group C (Vital Signs >70%) - Median + Indicator")

group_C_exists = [c for c in group_C_vital if c in df_train.columns]
print(f"사용 가능한 변수: {len(group_C_exists)}개 / {len(group_C_vital)}개")

for col in group_C_exists:
    median_val = df_train[col].median()
    df_train[col] = df_train[col].fillna(median_val)
    df_test[col] = df_test[col].fillna(median_val)

    print(f"   - {col}: median={median_val:.2f}")

print(f"✅ {len(group_C_exists)}개 값 + indicator 사용")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 6-4. Group D: 특수검사 (Indicator only)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n[6-4] Group D (특수검사 >70%) - Indicator만 사용")

group_D_exists = [c for c in group_D_special if c in df_train.columns]
print(f"사용 가능한 변수: {len(group_D_exists)}개 / {len(group_D_special)}개")

print(f"✅ {len(group_D_exists)}개 indicator만 사용 (원본 값 제외)")



[6] 결측치 대체

[6-1] Group A (수치형) - Median Imputation
사용 가능한 변수: 13개 / 13개
✅ 수치형 13개 처리 완료

[6-1-b] Group A (범주형) - GENDER Encoding
✅ GENDER: M=1, F=0 + mode imputation 완료

[6-2] Group B (중결측 30-70%) - MICE
사용 가능한 변수: 8개 / 8개
⏳ MICE 학습 중...
✅ 8개 변수 처리 완료

[6-3] Group C (Vital Signs >70%) - Median + Indicator
사용 가능한 변수: 7개 / 7개
   - HEART_RATE: median=86.75
   - SBP: median=120.50
   - DBP: median=67.00
   - MAP: median=80.00
   - RESP_RATE: median=19.00
   - TEMP: median=37.00
   - SPO2: median=98.00
✅ 7개 값 + indicator 사용

[6-4] Group D (특수검사 >70%) - Indicator만 사용
사용 가능한 변수: 5개 / 5개
✅ 5개 indicator만 사용 (원본 값 제외)


In [ ]:
# =============================================================================
# 8. 최종 변수 선택
# =============================================================================
print("\n" + "=" * 80)
print("[8] 최종 변수 선택")
print("=" * 80)

# 최종 사용 변수 목록
final_features = []

# Group A: 값
final_features.extend([col for col in group_A_exists])

# Group B: 값
final_features.extend([col for col in group_B_exists])

# Group C: 값 + Indicator
final_features.extend([col for col in group_C_exists])
final_features.extend([f'{col}_measured' for col in group_C_exists
                       if f'{col}_measured' in df_train.columns])

# Group D: Indicator만 (값 제외!)
final_features.extend([f'{col}_measured' for col in group_D_exists
                       if f'{col}_measured' in df_train.columns])

# Group E: Indicator만
final_features.extend([f'{col}_measured' for col in group_E_exclude
                       if f'{col}_measured' in df_train.columns])


# 최종 데이터프레임
X_train = df_train[final_features].copy()
X_test = df_test[final_features].copy()

print(f"\n✅ 최종 변수 선택 완료:")
print(f"   - Group A (저결측): {len(group_A_exists)}개 값")
print(f"   - Group B (중결측): {len(group_B_exists)}개 값")
print(f"   - Group C (Vital): {len(group_C_exists)}개 값 + {len(group_C_exists)}개 indicator")
print(f"   - Group D (특수검사): {len(group_D_exists)}개 indicator만 ⚠️")
print(f"   - Group E (제외): {len([col for col in group_E_exclude if col in df.columns])}개 indicator")

print(f"\n   총 {len(final_features)}개 변수")


[8] 최종 변수 선택


NameError: name 'group_A_exists' is not defined

In [ ]:
# =============================================================================
# 9. 결측치 확인
# =============================================================================
print("\n" + "=" * 80)
print("[9] 결측치 확인")
print("=" * 80)

train_missing = X_train.isnull().sum().sum()
test_missing = X_test.isnull().sum().sum()

print(f"Train 결측치: {train_missing}개")
print(f"Test 결측치: {test_missing}개")

if train_missing > 0 or test_missing > 0:
    print("\n⚠️ 결측치가 남아있습니다!")
    print("\n결측이 있는 변수:")
    for col in X_train.columns:
        train_null = X_train[col].isnull().sum()
        test_null = X_test[col].isnull().sum()
        if train_null > 0 or test_null > 0:
            print(f"   - {col}: Train={train_null}, Test={test_null}")
else:
    print("✅ 모든 결측치가 처리되었습니다!")

In [ ]:
# =============================================================================
# 10. 최종 데이터 저장
# =============================================================================
print("\n" + "=" * 80)
print("[10] 최종 데이터 저장")
print("=" * 80)

# Train 데이터
train_final = X_train.copy()
train_final['SEPSIS'] = y_train_sepsis.values
if y_mortality is not None:
    train_final['HOSPITAL_EXPIRE_FLAG'] = y_train_mortality.values

# Test 데이터
test_final = X_test.copy()
test_final['SEPSIS'] = y_test_sepsis.values
if y_mortality is not None:
    test_final['HOSPITAL_EXPIRE_FLAG'] = y_test_mortality.values

# CSV 저장
train_final.to_csv('train_preprocessed.csv', index=False, encoding='utf-8-sig')
test_final.to_csv('test_preprocessed.csv', index=False, encoding='utf-8-sig')

print("✅ 전처리 완료 데이터 저장:")
print(f"   - train_preprocessed.csv: {train_final.shape}")
print(f"   - test_preprocessed.csv: {test_final.shape}")



In [ ]:
# =============================================================================
# 11. 요약 출력
# =============================================================================
print("\n" + "=" * 80)
print("📊 전처리 요약")
print("=" * 80)

summary = f"""
원본 데이터: {df.shape}
  → Train: {len(df_train):,}명 (80%)
  → Test: {len(df_test):,}명 (20%)

변수 구성:
  Group A (저결측 <30%): {len(group_A_exists)}개 → Median
  Group B (중결측 30-70%): {len(group_B_exists)}개 → MICE
  Group C (Vital Signs): {len(group_C_exists)}개 값 + {len(group_C_exists)}개 indicator
  Group D (특수검사): {len(group_D_exists)}개 indicator만 (값 제외) ⚠️
  파생 변수: {len([col for col in derived_features if col in X_train.columns])}개

  총 {len(final_features)}개 변수

Target:
  패혈증 (SEPSIS): Train {y_train_sepsis.mean()*100:.2f}%, Test {y_test_sepsis.mean()*100:.2f}%
"""

if y_mortality is not None:
    summary += f"  사망 (HOSPITAL_EXPIRE_FLAG): Train {y_train_mortality.mean()*100:.2f}%, Test {y_test_mortality.mean()*100:.2f}%"

print(summary)

print("\n" + "=" * 80)
print("✅ 전처리 완료!")
print("=" * 80)

print("""
다음 단계:
1. train_preprocessed.csv, test_preprocessed.csv 다운로드
2. 모델 학습 (Gradient Boosting, Random Forest 등)
3. Threshold 최적화 (Recall=90% 목표)
4. 성능 평가 및 Feature Importance 분석
""")

In [ ]:
data = pd.read_csv(f'{base_path}cohort_csv/data.csv')

In [ ]:
data = df.reset_index(drop=True)

In [ ]:
train_data = df.iloc[train_idx]

In [ ]:
train_data.to_csv("train.data.csv", index=False)

In [ ]:
train_data.head()

In [ ]:
df.shape